# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [1]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate transformers peft
!pip install -q -U \
    langchain \
    langchain-community \
    langchain-chroma \
    langchain-huggingface \
    sentence-transformers \
    pandas
!pip install -q rouge-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 94.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 223.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.

In [2]:
# YOUR CODE HERE
import re
import json
import torch

from peft import PeftModel

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BEST_LORA_PATH = "./intent_lora_best"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load LoRA adapter and merge it into the base model
merged_model = PeftModel.from_pretrained(
    base_model,
    BEST_LORA_PATH
).merge_and_unload()

merged_model.eval()

print("Best LoRA checkpoint loaded and merged.")

INTENT_TO_SOP = {
    "track_order": "# Track Order",
    "cancel_order": "# Cancel Order",
    "refund_not_received": "# Refund Not Received",
    "request_refund": "# Refund",
    "return_order": "# Return",
    "exchange_order": "# Exchange",
    "order_status": "# Order Status",
    "delivery_time": "# Delivery Time",
    "change_shipping_address": "# Change Shipping Address",
    "payment_failed": "# Payment Failed",
    "payment_methods": "# Payment Methods",
    "invoice": "# Invoice",
    "damaged_product": "# Damaged Product",
    "wrong_item": "# Wrong Item",
    "missing_item": "# Missing Item",
    "product_information": "# Product Information",
    "discount": "# Discount",
    "promo_code": "# Promo Code",
    "account_issue": "# Account",
    "login_issue": "# Login",
    "password_reset": "# Password Reset",
    "contact_customer_service": "# Customer Service",
    "shipping_cost": "# Shipping Cost",
    "international_shipping": "# International Shipping",
    "domestic_shipping": "# Domestic Shipping",
    "subscription": "# Subscription",
    "other": "# Other"
}

def intent_to_search_string(intent, raw_query):
    """
    Convert structured intent into an SOP retrieval query.
    Fall back to the original user query if no mapping exists.
    """

    intent = intent.strip().lower()

    if intent in INTENT_TO_SOP:
        return INTENT_TO_SOP[intent]

    return raw_query

SYSTEM_PROMPT = """You are an intent classification router for a customer-support system.

Classify the user's request into exactly one supported intent.

Return ONLY valid JSON in this format:
{"intent": "intent_name", "category": "category_name"}

Do not provide explanations.
Do not provide additional text.
Do not use Markdown.

Examples:

User: Where is my order?
Assistant: {"intent": "track_order", "category": "ORDER"}

User: I want to cancel my order.
Assistant: {"intent": "cancel_order", "category": "ORDER"}

User: How long will my domestic order take?
Assistant: {"intent": "delivery_time", "category": "SHIPPING"}

User: I haven't received my refund yet.
Assistant: {"intent": "refund_not_received", "category": "REFUND"}
"""

def classify_intent(user_query):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(merged_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    raw_output = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return raw_output

def extract_intent_json(raw_output):

    # First try direct JSON parsing
    try:
        result = json.loads(raw_output)

        if "intent" in result:
            return result

    except json.JSONDecodeError:
        pass

    # Safety net: extract the first JSON object
    match = re.search(r'\{.*?\}', raw_output, re.DOTALL)

    if match:

        try:
            result = json.loads(match.group(0))

            if "intent" in result:
                return result

        except json.JSONDecodeError:
            pass

    return None

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Best LoRA checkpoint loaded and merged.


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [3]:
# YOUR CODE HERE
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from google.colab import drive
drive.mount('/content/drive')

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_DIR = "/content/drive/MyDrive/Colab Notebooks/HYBRID-RAG-AND-FINE-TUNING-ASSIGNMENT/OUPUT/chroma_db"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vector_db = Chroma(
    collection_name="sop_documents",
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings
)

print("✓ ChromaDB reloaded")
print("Documents in index:", vector_db._collection.count())

def retrieve_sop(user_query):

    raw_output = classify_intent(user_query)

    intent_data = extract_intent_json(raw_output)

    if intent_data is not None:

        intent = intent_data.get("intent", "").lower()

        search_string = intent_to_search_string(
            intent,
            user_query
        )

    else:

        # JSON parsing failed → use original query
        intent = None
        search_string = user_query

    docs = vector_db.similarity_search(
        search_string,
        k=1
    )

    return {
        "raw_router_output": raw_output,
        "intent_data": intent_data,
        "intent": intent,
        "search_string": search_string,
        "documents": docs
    }


def generate_final_answer(user_query, retrieved_context):

    final_prompt = f"""You are a customer-support assistant.

Answer the user's question strictly using the SOP provided below.

Do not invent information.
Do not add policies that are not present in the SOP.
If the SOP does not contain enough information, say that the available policy does not provide enough information.

SOP:
{retrieved_context}

User question:
{user_query}
"""

    messages = [
        {
            "role": "system",
            "content": final_prompt
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(merged_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

def hybrid_rag(user_query):

    # ---------------------------------------------
    # Step 1: Fine-tuned model predicts intent
    # ---------------------------------------------
    raw_router_output = classify_intent(user_query)

    # ---------------------------------------------
    # Step 2: Extract JSON
    # ---------------------------------------------
    intent_data = extract_intent_json(raw_router_output)

    if intent_data:

        intent = intent_data.get("intent", "").lower()
        category = intent_data.get("category", "")

        # -----------------------------------------
        # Step 3: Intent → SOP search string
        # -----------------------------------------
        search_string = intent_to_search_string(
            intent,
            user_query
        )

    else:

        intent = None
        category = None

        # -----------------------------------------
        # Fallback
        # -----------------------------------------
        search_string = user_query

    # ---------------------------------------------
    # Step 4: Vector search
    # ---------------------------------------------
    docs = vector_db.similarity_search(
        search_string,
        k=1
    )

    # ---------------------------------------------
    # Step 5: Build SOP context
    # ---------------------------------------------
    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # ---------------------------------------------
    # Step 6: Generate final answer
    # ---------------------------------------------
    answer = generate_final_answer(
        user_query,
        context
    )

    return {
        "query": user_query,
        "router_output": raw_router_output,
        "intent": intent,
        "category": category,
        "search_string": search_string,
        "retrieved_sop": context,
        "answer": answer
    }

test_query = "Where is my order? I placed it a few days ago."

result = hybrid_rag(test_query)

print("=" * 70)
print("USER QUERY")
print("=" * 70)
print(result["query"])

print("\n" + "=" * 70)
print("ROUTER OUTPUT")
print("=" * 70)
print(result["router_output"])

print("\n" + "=" * 70)
print("PREDICTED INTENT")
print("=" * 70)
print(result["intent"])

print("\n" + "=" * 70)
print("CATEGORY")
print("=" * 70)
print(result["category"])

print("\n" + "=" * 70)
print("SOP SEARCH STRING")
print("=" * 70)
print(result["search_string"])

print("\n" + "=" * 70)
print("RETRIEVED SOP")
print("=" * 70)
print(result["retrieved_sop"])

print("\n" + "=" * 70)
print("FINAL ANSWER")
print("=" * 70)
print(result["answer"])

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ ChromaDB reloaded
Documents in index: 13
USER QUERY
Where is my order? I placed it a few days ago.

ROUTER OUTPUT
{"intent": "track_order", "category": "ORDER"}

PREDICTED INTENT
track_order

CATEGORY
ORDER

SOP SEARCH STRING
# Track Order

RETRIEVED SOP
# Order Tracking

## Scope
Helps customers locate an order and interpret its status. Orders that have
clearly exceeded their delivery window follow the shipping delays procedure.

## Order States
- **Processing:** payment confirmed, not yet packed.
- **Preparing to ship:** packed, awaiting carrier pickup.
- **In transit:** carrier has the parcel; tracking events update as it moves.
- **Out for delivery:** on the vehicle for delivery that day.
- **Delivered:** carrier marked the parcel delivered.
- **Exception:** address issue, failed delivery, or held parcel needing
  action.

## Finding The Order
Locate the order by its identifier and the registered email. Provide the
tracking number and carrier link. Tracking can take up to 24 hour

### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [18]:
# YOUR CODE HERE
import re
import json
import difflib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# df_test must come from Notebook 2's original 80/10/10 split.
# Do NOT recreate the split here using the training data.

TEST_FILE = "/content/drive/MyDrive/Colab Notebooks/HYBRID-RAG-AND-FINE-TUNING-ASSIGNMENT/OUPUT/test_data.csv"

df_test = pd.read_csv(TEST_FILE)

print("Test samples:", len(df_test))
print(df_test[["instruction", "intent", "category"]].head())

adversarial_pattern = re.compile(
    r"\b(still|never|terrible|frustrated)\b",
    flags=re.IGNORECASE
)

df_adversarial = df_test[
    df_test["instruction"].astype(str).str.contains(
        adversarial_pattern,
        regex=True,
        na=False
    )
].copy()

print("Full test set:", len(df_test))
print("Adversarial subset:", len(df_adversarial))

display(
    df_adversarial[
        ["instruction", "intent", "category"]
    ].head(10)
)

print(
    df_test[
        df_test["instruction"]
        .astype(str)
        .str.contains(
            "still|never|terrible|frustrated",
            case=False,
            regex=True,
            na=False
        )
    ][["instruction", "intent"]].head(20)
)

def is_valid_intent_json(raw_output):

    if not isinstance(raw_output, str):
        return False

    try:
        result = json.loads(raw_output)

        return (
            isinstance(result, dict)
            and "intent" in result
            and "category" in result
            and isinstance(result["intent"], str)
            and isinstance(result["category"], str)
        )

    except (json.JSONDecodeError, TypeError):
        return False
def evaluate_router(df, description="Dataset"):

    columns = [
        "instruction",
        "true_intent",
        "raw_output",
        "format_adherent",
        "predicted_intent"
    ]

    # Handle empty dataset
    if len(df) == 0:
        print(f"{description}: No samples found.")
        return pd.DataFrame(columns=columns)

    records = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Evaluating {description}"
    ):

        query = str(row["instruction"])
        true_intent = str(row["intent"]).strip().lower()

        raw_output = classify_intent(query)

        # Check whether output is valid JSON
        format_ok = is_valid_intent_json(raw_output)

        # Extract JSON
        parsed = extract_intent_json(raw_output)

        if parsed is not None:
            predicted_intent = str(
                parsed.get("intent", "")
            ).strip().lower()
        else:
            predicted_intent = ""

        records.append({
            "instruction": query,
            "true_intent": true_intent,
            "raw_output": raw_output,
            "format_adherent": bool(format_ok),
            "predicted_intent": predicted_intent
        })

    return pd.DataFrame(records)

def exact_match(pred, true):

    return (
        str(pred).strip().lower()
        ==
        str(true).strip().lower()
    )

def normalize_intent(intent):

    intent = str(intent).lower().strip()

    # Treat underscores/spaces consistently
    intent = re.sub(r"[\s\-]+", "_", intent)

    return intent


def fuzzy_match(pred, true, threshold=0.80):

    pred = normalize_intent(pred)
    true = normalize_intent(true)

    if not pred:
        return False

    similarity = difflib.SequenceMatcher(
        None,
        pred,
        true
    ).ratio()

    return similarity >= threshold

def calculate_router_metrics(results):

    # Make sure format_adherent is Boolean
    format_values = results["format_adherent"].astype(bool)

    # Format adherence
    format_adherence_pct = (
        format_values.mean() * 100
    )

    # Exact intent match
    exact_match_pct = np.mean([
        exact_match(pred, true)
        for pred, true in zip(
            results["predicted_intent"],
            results["true_intent"]
        )
    ]) * 100

    # Fuzzy intent match
    fuzzy_match_pct = np.mean([
        fuzzy_match(pred, true)
        for pred, true in zip(
            results["predicted_intent"],
            results["true_intent"]
        )
    ]) * 100

    return {
        "Format Adherence (%)": format_adherence_pct,
        "Exact Match (%)": exact_match_pct,
        "Fuzzy Match (%)": fuzzy_match_pct
    }

test_results = evaluate_router(
    df_test,
    "Held-out Test Set"
)

print("router test results:", test_results);

test_metrics = calculate_router_metrics(
    test_results
)

print("===== HELD-OUT TEST RESULTS =====")

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.2f}%")

adversarial_results = evaluate_router(
    df_adversarial,
    "Adversarial Subset"
)

adversarial_metrics = calculate_router_metrics(
    adversarial_results
)

print("===== ADVERSARIAL RESULTS =====")
print("Adversarial subset size:", len(df_adversarial))
print("Adversarial results size:", len(adversarial_results))

print("\ndf_adversarial columns:")
print(df_adversarial.columns.tolist())

print("\nadversarial_results columns:")
print(adversarial_results.columns.tolist())


for metric, value in adversarial_metrics.items():
    print(f"{metric}: {value:.2f}%")

evaluation_summary = pd.DataFrame([
    {
        "Dataset": "Held-out Test",
        **test_metrics
    },
    {
        "Dataset": "Adversarial",
        **adversarial_metrics
    }
])

display(evaluation_summary)


errors = test_results[
    ~test_results.apply(
        lambda row: exact_match(
            row["predicted_intent"],
            row["true_intent"]
        ),
        axis=1
    )
]

print("Incorrect predictions:", len(errors))

display(
    errors[
        [
            "instruction",
            "true_intent",
            "predicted_intent",
            "raw_output"
        ]
    ].head(20)
)

smooth = SmoothingFunction().method1

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

synthesis_records = []

for _, row in tqdm(
    df_test.iterrows(),
    total=len(df_test),
    desc="Evaluating final synthesis"
):

    REFERENCE_COLUMN = "intent"

    assert REFERENCE_COLUMN in df_test.columns, (
        f"Reference column '{REFERENCE_COLUMN}' not found. "
        f"Available columns: {df_test.columns.tolist()}"
    )

    print("Using reference column:", REFERENCE_COLUMN)

    query = str(row["instruction"])
    reference = str(row[REFERENCE_COLUMN])

    result = hybrid_rag(query)

    prediction = result["answer"]

    scores = rouge.score(
        reference,
        prediction
    )

    bleu = sentence_bleu(
        [reference.split()],
        prediction.split(),
        smoothing_function=smooth
    )

    synthesis_records.append({
        "instruction": query,
        "reference": reference,
        "prediction": prediction,
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
        "bleu": bleu
    })

synthesis_results = pd.DataFrame(
    synthesis_records
)

display(synthesis_results.head())

synthesis_metrics = {
    "baseline_rouge1": synthesis_results["rouge1"].mean(),
    "ROUGE-2": synthesis_results["rouge2"].mean(),
    "baseline_rougeL": synthesis_results["rougeL"].mean(),
    "baseline_bleu": synthesis_results["bleu"].mean()
}

print("===== FINAL SYNTHESIS METRICS =====")

for metric, score in synthesis_metrics.items():
    print(f"{metric}: {score:.4f}")

test_results.to_csv(
    "test_intent_evaluation.csv",
    index=False
)

adversarial_results.to_csv(
    "adversarial_intent_evaluation.csv",
    index=False
)

evaluation_summary.to_csv(
    "intent_evaluation_summary.csv",
    index=False
)

if "synthesis_results" in globals():
    synthesis_results.to_csv(
        "synthesis_evaluation.csv",
        index=False
    )

print("Evaluation artifacts saved.")

Test samples: 391
                                         instruction                   intent  \
0               need help to shop several of ur item              place_order   
1  i need assistance trying to update the shippin...  change_shipping_address   
2  I do not know how to check the cancellation  p...   check_cancellation_fee   
3      want assistance to see the payment modalities    check_payment_methods   
4  how can I get information about opening a stan...           create_account   

   category  
0     ORDER  
1  SHIPPING  
2    CANCEL  
3   PAYMENT  
4   ACCOUNT  
Full test set: 391
Adversarial subset: 0


/tmp/ipykernel_2775/204974324.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_test["instruction"].astype(str).str.contains(


,instruction,intent,category


Empty DataFrame
Columns: [instruction, intent]
Index: []


Evaluating Held-out Test Set:   0%|          | 0/391 [00:00<?, ?it/s]

...

,instruction,true_intent,predicted_intent,raw_output
0,need help to shop several of ur item,place_order,shop_multiple_items,"{""intent"": ""shop_multiple_items"", ""category"": ..."
1,i need assistance trying to update the shippin...,change_shipping_address,update_shipping_address,"{""intent"": ""update_shipping_address"", ""categor..."
2,I do not know how to check the cancellation p...,check_cancellation_fee,check_cancellation_penalty,"{""intent"": ""check_cancellation_penalty"", ""cate..."
3,want assistance to see the payment modalities,check_payment_methods,payment_modality,"{""intent"": ""payment_modality"", ""category"": ""HE..."
4,how can I get information about opening a stan...,create_account,account_opening_information,"{""intent"": ""account_opening_information"", ""cat..."
5,I do not know how to see the status of order {...,track_order,view_order_status,"{""intent"": ""view_order_status"", ""category"": ""O..."
6,need help editing m address,change_shipping_address,address_editing,"{""intent"": ""address_editing"", ""category"": ""USER""}"
7,how do I see at what time I can call customer ...,contact_customer_service,contact_support,"{""intent"": ""contact_support"", ""category"": ""HELP""}"
8,using standard accouny,switch_account,account_settings,"{""intent"": ""account_settings"", ""category"": ""AC..."
9,I cannot afford order {{Order Number}},cancel_order,order_refund_request,"{""intent"": ""order_refund_request"", ""category"":..."


Evaluating final synthesis:   0%|          | 0/391 [00:00<?, ?it/s]

Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using reference column: intent
Using re

,instruction,reference,prediction,rouge1,rouge2,rougeL,bleu
0,need help to shop several of ur item,place_order,"I'm sorry, but I don't understand your request...",0.000000,0.000000,0.000000,0.0
1,i need assistance trying to update the shippin...,change_shipping_address,To assist you in updating the shipping address...,0.062500,0.021277,0.062500,0.0
2,I do not know how to check the cancellation p...,check_cancellation_fee,The cancellation options you have mentioned in...,0.018519,0.000000,0.018519,0.0
3,want assistance to see the payment modalities,check_payment_methods,The store accepts major credit and debit cards...,0.090909,0.047619,0.090909,0.0
4,how can I get information about opening a stan...,create_account,To obtain information about opening a standard...,0.025316,0.000000,0.025316,0.0


===== FINAL SYNTHESIS METRICS =====
baseline_rouge1: 0.0375
ROUGE-2: 0.0056
baseline_rougeL: 0.0357
baseline_bleu: 0.0000
Evaluation artifacts saved.


#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [19]:
# YOUR CODE HERE
import pandas as pd
import numpy as np

TEST_FILE = "/content/drive/MyDrive/Colab Notebooks/HYBRID-RAG-AND-FINE-TUNING-ASSIGNMENT/OUPUT/rouge_bleu_results.csv"

v1 = pd.read_csv(TEST_FILE)# or the naive_rag_* columns from this file
v2 = synthesis_metrics             # your V2 (Hybrid RAG) per-query or mean scores

metrics = ["baseline_rouge1", "baseline_rougeL", "baseline_bleu"]
rows = []
for m in metrics:
    v1_score = v1[m].mean() if isinstance(v1, pd.DataFrame) else v1[m]
    v2_score = v2[m].mean() if isinstance(v2, pd.DataFrame) else v2[m]
    rows.append({
        "metric": m,
        "v1_naive_rag": v1_score,
        "v2_hybrid_rag": v2_score,
        "improvement_%": (v2_score - v1_score) / v1_score * 100,
    })

comparison = pd.DataFrame(rows)
print(comparison.round(4))

            metric  v1_naive_rag  v2_hybrid_rag  improvement_%
0  baseline_rouge1        0.1271         0.0375       -70.4974
1  baseline_rougeL        0.0711         0.0357       -49.7755
2    baseline_bleu        0.0008         0.0000       -97.3357


### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [28]:
# YOUR CODE HERE
import pandas as pd
import numpy as np
import json
import re
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ============================================================
# 1. Evaluation dataset
# ============================================================

# Full held-out test split
#df_eval = df_test.copy()

# For a quick test, use:
df_eval = df_test.head(50).copy()

print(f"Evaluation samples: {len(df_eval)}")


# ============================================================
# 2. Identify the reference-answer column
# ============================================================

# Use the SOP-grounded reference column created earlier.
# Change this if your notebook uses a different column name.
REFERENCE_COLUMN = "intent"

if REFERENCE_COLUMN not in df_eval.columns:
    print("Available columns:")
    print(df_eval.columns.tolist())
    raise KeyError(
        f"'{REFERENCE_COLUMN}' not found in df_test. "
        "Set REFERENCE_COLUMN to the column containing the SOP-grounded reference answers."
    )

print("Reference column:", REFERENCE_COLUMN)


# ============================================================
# 3. Scoring functions
# ============================================================

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

smooth = SmoothingFunction().method1


def calculate_scores(prediction, reference):
    """Calculate ROUGE-1, ROUGE-2, ROUGE-L and BLEU."""

    prediction = str(prediction).strip()
    reference = str(reference).strip()

    rouge_scores = rouge.score(reference, prediction)

    reference_tokens = reference.lower().split()
    prediction_tokens = prediction.lower().split()

    if len(prediction_tokens) == 0:
        bleu = 0.0
    else:
        bleu = sentence_bleu(
            [reference_tokens],
            prediction_tokens,
            smoothing_function=smooth
        )

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "BLEU": bleu
    }

merged_model = PeftModel.from_pretrained(
    base_model,
    "./intent_lora_best"
).merge_and_unload()

generation_model = merged_model

# ============================================================
# 4. Baseline generation
# ============================================================

def generate_baseline(query):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a customer-support assistant. "
                "Answer the customer's question clearly and concisely."
            )
        },
        {
            "role": "user",
            "content": query
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(generation_model.device)

    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


# ============================================================
# 5. Naive RAG
# ============================================================

def naive_rag(query, k=1):

    docs = vector_db.similarity_search(query, k=k)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a customer-support assistant. "
                "Answer strictly using the SOP information provided below. "
                "Do not invent policies or information.\n\n"
                f"SOP:\n{context}"
            )
        },
        {
            "role": "user",
            "content": query
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(generation_model.device)

    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


# ============================================================
# 6. Hybrid RAG
# ============================================================

# Make sure your previously defined hybrid_rag() is available.
# It should:
#   1. classify the intent
#   2. map intent -> SOP
#   3. retrieve the relevant SOP
#   4. synthesize the final answer

def hybrid_answer(query):
    result = hybrid_rag(query)

    # hybrid_rag() from the previous notebook returns a dictionary
    # containing the final answer.
    if isinstance(result, dict):
        return result.get("answer", "")

    return str(result)


# ============================================================
# 7. Run all three architectures
# ============================================================

results = []

for i, (_, row) in enumerate(df_eval.iterrows(), start=1):

    query = str(row["instruction"])
    reference = str(row[REFERENCE_COLUMN])

    print(f"\rProcessing {i}/{len(df_eval)}", end="")

    # -------------------------
    # Baseline
    # -------------------------
    baseline_answer = generate_baseline(query)

    baseline_scores = calculate_scores(
        baseline_answer,
        reference
    )

    # -------------------------
    # Naive RAG
    # -------------------------
    naive_answer = naive_rag(query)

    naive_scores = calculate_scores(
        naive_answer,
        reference
    )

    # -------------------------
    # Hybrid RAG
    # -------------------------
    hybrid_answer_text = hybrid_answer(query)

    hybrid_scores = calculate_scores(
        hybrid_answer_text,
        reference
    )

    # -------------------------
    # Store row
    # -------------------------
    results.append({
        "row_id": row.name,
        "instruction": query,
        "reference": reference,

        "baseline_answer": baseline_answer,
        "baseline_rouge1": baseline_scores["ROUGE-1"],
        "baseline_rouge2": baseline_scores["ROUGE-2"],
        "baseline_rougeL": baseline_scores["ROUGE-L"],
        "baseline_bleu": baseline_scores["BLEU"],

        "naive_rag_answer": naive_answer,
        "naive_rag_rouge1": naive_scores["ROUGE-1"],
        "naive_rag_rouge2": naive_scores["ROUGE-2"],
        "naive_rag_rougeL": naive_scores["ROUGE-L"],
        "naive_rag_bleu": naive_scores["BLEU"],

        "hybrid_rag_answer": hybrid_answer_text,
        "hybrid_rag_rouge1": hybrid_scores["ROUGE-1"],
        "hybrid_rag_rouge2": hybrid_scores["ROUGE-2"],
        "hybrid_rag_rougeL": hybrid_scores["ROUGE-L"],
        "hybrid_rag_bleu": hybrid_scores["BLEU"]
    })

print("\nEvaluation complete.")


# ============================================================
# 8. Per-row results
# ============================================================

comparative_results = pd.DataFrame(results)

comparative_results.to_csv(
    "Comparative_Results_Full.csv",
    index=False
)

print(
    f"Saved Comparative_Results_Full.csv "
    f"({len(comparative_results)} rows)"
)

display(comparative_results.head())


# ============================================================
# 9. Aggregate summary
# ============================================================

summary = []

metric_pairs = [
    ("ROUGE-1", "rouge1"),
    ("ROUGE-2", "rouge2"),
    ("ROUGE-L", "rougeL"),
    ("BLEU", "bleu")
]

for metric_name, suffix in metric_pairs:

    baseline = comparative_results[
        f"baseline_{suffix}"
    ].mean()

    naive = comparative_results[
        f"naive_rag_{suffix}"
    ].mean()

    hybrid = comparative_results[
        f"hybrid_rag_{suffix}"
    ].mean()

    summary.append({
        "Metric": metric_name,
        "Baseline": baseline,
        "Naive RAG": naive,
        "Hybrid RAG": hybrid,
        "Baseline_to_Naive_Improvement_%":
            ((naive - baseline) / baseline * 100)
            if baseline != 0 else np.nan,
        "Naive_to_Hybrid_Improvement_%":
            ((hybrid - naive) / naive * 100)
            if naive != 0 else np.nan,
        "Baseline_to_Hybrid_Improvement_%":
            ((hybrid - baseline) / baseline * 100)
            if baseline != 0 else np.nan
    })

summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    "Comparative_Results_Summary.csv",
    index=False
)

print("Saved Comparative_Results_Summary.csv")

display(summary_df)

Evaluation samples: 50
Reference column: intent
Processing 1/50

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Processing 50/50
Evaluation complete.
Saved Comparative_Results_Full.csv (50 rows)


,row_id,instruction,reference,baseline_answer,baseline_rouge1,baseline_rouge2,baseline_rougeL,baseline_bleu,naive_rag_answer,naive_rag_rouge1,naive_rag_rouge2,naive_rag_rougeL,naive_rag_bleu,hybrid_rag_answer,hybrid_rag_rouge1,hybrid_rag_rouge2,hybrid_rag_rougeL,hybrid_rag_bleu
0,0,need help to shop several of ur item,place_order,"I'm sorry, I am not sure what you want to ask....",0.029851,0.000000,0.029851,0.0,"I'm sorry, but I am unable to assist with that...",0.020202,0.000000,0.020202,0,"I'm sorry, but I don't know how to assist you ...",0.000000,0.000000,0.000000,0
1,1,i need assistance trying to update the shippin...,change_shipping_address,I'm here to help you. Can you please provide m...,0.000000,0.000000,0.000000,0.0,"Sure, I'll help you track your order. Please p...",0.057971,0.029851,0.057971,0,The user needs help updating the shipping addr...,0.059406,0.020202,0.059406,0
2,2,I do not know how to check the cancellation p...,check_cancellation_fee,"To check the cancellation fees, you can contac...",0.074074,0.025316,0.074074,0.0,Please note that you can check your subscripti...,0.041667,0.000000,0.041667,0,The SMS notification will inform you about the...,0.061856,0.021053,0.061856,0
3,3,want assistance to see the payment modalities,check_payment_methods,"I'm sorry, I can't see the payment options as ...",0.105263,0.000000,0.052632,0.0,"Sure, I'll help you understand the payment opt...",0.056604,0.019231,0.037736,0,The store accepts major credit and debit cards...,0.039216,0.020000,0.039216,0
4,4,how can I get information about opening a stan...,create_account,To check the requirements for opening a standa...,0.035088,0.000000,0.035088,0.0,To check your eligibility to open a new accoun...,0.019608,0.000000,0.019608,0,The assistance you seek is related to the crea...,0.040000,0.000000,0.040000,0


Saved Comparative_Results_Summary.csv


,Metric,Baseline,Naive RAG,Hybrid RAG,Baseline_to_Naive_Improvement_%,Naive_to_Hybrid_Improvement_%,Baseline_to_Hybrid_Improvement_%
0,ROUGE-1,0.048168,0.031192,0.027390,-35.243278,-12.190352,-43.137351
1,ROUGE-2,0.008304,0.003870,0.004361,-53.397304,12.689545,-47.483633
2,ROUGE-L,0.041613,0.028630,0.026218,-31.198972,-8.425053,-36.995495
3,BLEU,0.000039,0.000000,0.000000,-100.000000,NaN,-100.000000


In [ ]:
# YOUR CODE HERE


---
## END-OF-NOTEBOOK CHECKLIST (FINAL)

> **IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [ ] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [ ] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [ ] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [ ] **4.5** All 3 architectures scored with SOP-grounded references
- [ ] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [ ] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**